# Aggregate-only exploration

This notebook is intentionally thin. It calls the package loaders and displays aggregate status counts and feature summaries only; it never displays raw rows or quasi-identifying records. Run it from the repository root after placing the authorized raw CSV in `data/raw/`.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from credit_default.data import load_raw_data, prepare_supervised_data

raw = load_raw_data(repo_root / 'data/raw/loan.csv')
X, y = prepare_supervised_data(raw)
print({'raw_rows': len(raw), 'labeled_rows': len(y), 'raw_columns': raw.shape[1], 'feature_columns_before_encoding': X.shape[1]})

In [ ]:
import matplotlib.pyplot as plt

status_counts = raw['loan_status'].fillna('(blank)').value_counts(dropna=False).sort_values()
status_counts.plot(kind='barh', color='#0072B2', figsize=(7, 4))
plt.title('Aggregate loan-status counts')
plt.xlabel('Rows')
plt.ylabel('Status')
plt.grid(axis='x', alpha=0.2)
plt.tight_layout()

In [ ]:
summary = {
    'positive_rate': float(y.mean()),
    'class_counts': {str(int(k)): int(v) for k, v in y.value_counts().sort_index().items()},
    'missing_rates_top': raw.isna().mean().sort_values(ascending=False).head(8).round(3).to_dict(),
}
summary